# Results Analysis - Deep Performance Analytics

This notebook provides advanced analytics and visualizations for understanding strategy performance.

**Perfect for**: Creating publication-quality charts and comprehensive performance reports.

## What You'll Learn

1. How to create professional performance tear sheets
2. How to analyze returns distributions
3. How to compute advanced risk metrics
4. How to visualize portfolio composition over time
5. How to identify performance drivers

## Prerequisites

Complete the previous notebooks to understand the basics.

---

## Step 1: Setup and Run Backtest

In [ ]:
# Add parent directory to path
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

# Import tools
from Strategies.Registry import quick_strategy
from Backtest.MinimalBacktest import MinimalBacktest
import numpy as np
import pandas as pd
from datetime import date
from scipy import stats

# Import visualization
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from matplotlib.patches import Rectangle

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['font.size'] = 10

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

print("✓ Setup complete!")

In [ ]:
# Create mock market data
class SimpleMockMDP:
    """Mock market data provider."""
    
    def __init__(self, base_rate=5.0, carry_spread=0.10, seed=42):
        self.base_rate = base_rate
        self.carry_spread = carry_spread
        self._rng = np.random.RandomState(seed)
    
    def get_pricer(self, currency, as_of):
        return self
    
    def futures_price(self, contract):
        quarter_map = {'H': 0, 'M': 1, 'U': 2, 'Z': 3}
        quarter_code = contract[-2] if len(contract) >= 2 else 'H'
        quarter = quarter_map.get(quarter_code, 0)
        rate = self.base_rate + (quarter * self.carry_spread)
        price = 100.0 - rate
        noise = self._rng.normal(0, 0.01)
        return price + noise

# Setup and run backtest
mdp = SimpleMockMDP(base_rate=5.0, carry_spread=0.10, seed=42)
instruments = ['SFRZ4', 'SFRH5', 'SFRM5', 'SFRU5', 'SFRZ5']
start_date = date(2024, 6, 1)
end_date = date(2024, 12, 1)
dates = pd.date_range(start_date, end_date, freq='W').tolist()
dates = [d.date() if hasattr(d, 'date') else d for d in dates]

backtest = MinimalBacktest(mdp=mdp, risk_aversion=1.5, long_only=True, min_history=5)
result = backtest.run(contracts=instruments, dates=dates)

print("✓ Backtest complete!")
print(f"  Periods: {len(result.returns)}")
print(f"  Instruments: {len(instruments)}")

---

## Step 2: Calculate Advanced Metrics

Let's compute a comprehensive set of performance metrics:

In [ ]:
def calculate_metrics(returns):
    """Calculate comprehensive performance metrics."""
    
    # Basic metrics
    total_return = (1 + returns).prod() - 1
    ann_return = (1 + returns.mean()) ** 52 - 1  # Annualized
    ann_vol = returns.std() * np.sqrt(52)  # Annualized
    sharpe = ann_return / ann_vol if ann_vol > 0 else 0
    
    # Drawdown metrics
    cumulative = returns.cumsum()
    running_max = cumulative.cummax()
    drawdown = cumulative - running_max
    max_drawdown = drawdown.min()
    
    # Calmar ratio (return / max drawdown)
    calmar = ann_return / abs(max_drawdown) if max_drawdown < 0 else 0
    
    # Win rate and profit factor
    wins = returns[returns > 0]
    losses = returns[returns < 0]
    win_rate = len(wins) / len(returns) if len(returns) > 0 else 0
    avg_win = wins.mean() if len(wins) > 0 else 0
    avg_loss = losses.mean() if len(losses) > 0 else 0
    profit_factor = abs(wins.sum() / losses.sum()) if len(losses) > 0 and losses.sum() != 0 else 0
    
    # Distribution metrics
    skewness = stats.skew(returns)
    kurtosis = stats.kurtosis(returns)
    
    # Value at Risk (VaR) and Conditional VaR (CVaR)
    var_95 = np.percentile(returns, 5)
    cvar_95 = returns[returns <= var_95].mean()
    
    return {
        'Total Return': total_return,
        'Annualized Return': ann_return,
        'Annualized Volatility': ann_vol,
        'Sharpe Ratio': sharpe,
        'Max Drawdown': max_drawdown,
        'Calmar Ratio': calmar,
        'Win Rate': win_rate,
        'Average Win': avg_win,
        'Average Loss': avg_loss,
        'Profit Factor': profit_factor,
        'Skewness': skewness,
        'Kurtosis': kurtosis,
        'VaR (95%)': var_95,
        'CVaR (95%)': cvar_95
    }

metrics = calculate_metrics(result.returns)

print("Comprehensive Performance Metrics")
print("=" * 60)
print()

for metric, value in metrics.items():
    if 'Rate' in metric or 'Return' in metric or 'Drawdown' in metric or 'VaR' in metric or 'Win' in metric or 'Loss' in metric:
        print(f"{metric:.<35s} {value:>10.2%}")
    else:
        print(f"{metric:.<35s} {value:>10.3f}")

---

## Step 3: Create Performance Tear Sheet

A comprehensive visual summary of strategy performance:

In [ ]:
# Create comprehensive tear sheet
fig = plt.figure(figsize=(18, 14))
gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.4, wspace=0.3)

# Color scheme
primary_color = 'steelblue'
positive_color = 'green'
negative_color = 'red'

# 1. Cumulative Returns (top span)
ax1 = fig.add_subplot(gs[0, :])
cumulative = (1 + result.returns).cumprod()
ax1.fill_between(cumulative.index, 1, cumulative.values, alpha=0.3, color=primary_color)
ax1.plot(cumulative.index, cumulative.values, linewidth=2.5, color=primary_color, label='Strategy')
ax1.axhline(y=1.0, color='black', linestyle='--', alpha=0.5, linewidth=1)
ax1.set_title('Cumulative Returns', fontsize=16, fontweight='bold', pad=15)
ax1.set_ylabel('Growth of $1', fontsize=12)
ax1.legend(loc='best', fontsize=11)
ax1.grid(True, alpha=0.3)

# Add final value annotation
final_value = cumulative.iloc[-1]
ax1.annotate(f'${final_value:.3f}', 
             xy=(cumulative.index[-1], final_value),
             xytext=(10, 0), textcoords='offset points',
             fontsize=12, fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor=primary_color, linewidth=2))

# 2. Drawdown
ax2 = fig.add_subplot(gs[1, :])
cumulative_pct = result.returns.cumsum()
running_max = cumulative_pct.cummax()
drawdown = cumulative_pct - running_max
ax2.fill_between(drawdown.index, 0, drawdown.values, alpha=0.5, color=negative_color)
ax2.plot(drawdown.index, drawdown.values, linewidth=2, color=negative_color)
ax2.set_title('Drawdown', fontsize=16, fontweight='bold', pad=15)
ax2.set_ylabel('Drawdown', fontsize=12)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: '{:.1%}'.format(y)))
ax2.grid(True, alpha=0.3)

# Mark max drawdown
max_dd_idx = drawdown.idxmin()
max_dd_val = drawdown.min()
ax2.plot(max_dd_idx, max_dd_val, 'o', markersize=10, color='darkred', 
         markeredgecolor='black', markeredgewidth=2, label=f'Max: {max_dd_val:.2%}')
ax2.legend(loc='best', fontsize=11)

# 3. Returns Distribution
ax3 = fig.add_subplot(gs[2, 0])
returns_pct = result.returns * 100
n, bins, patches = ax3.hist(returns_pct, bins=20, alpha=0.7, color=primary_color, edgecolor='black')
# Color bars by sign
for i, patch in enumerate(patches):
    if bins[i] < 0:
        patch.set_facecolor(negative_color)
    else:
        patch.set_facecolor(positive_color)
ax3.axvline(returns_pct.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {returns_pct.mean():.2f}%')
ax3.axvline(0, color='black', linestyle='-', alpha=0.3, linewidth=1)
ax3.set_title('Returns Distribution', fontsize=14, fontweight='bold')
ax3.set_xlabel('Weekly Return (%)', fontsize=11)
ax3.set_ylabel('Frequency', fontsize=11)
ax3.legend(loc='best', fontsize=10)
ax3.grid(True, alpha=0.3, axis='y')

# 4. Monthly Returns Heatmap
ax4 = fig.add_subplot(gs[2, 1])
# Group returns by month
monthly_returns = result.returns.groupby(pd.Grouper(freq='M')).sum()
months = ['Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'][:len(monthly_returns)]
colors_monthly = [positive_color if r > 0 else negative_color for r in monthly_returns.values]
bars = ax4.bar(months, monthly_returns.values * 100, color=colors_monthly, alpha=0.7, edgecolor='black')
ax4.axhline(0, color='black', linestyle='-', alpha=0.3, linewidth=1)
ax4.set_title('Monthly Returns', fontsize=14, fontweight='bold')
ax4.set_ylabel('Return (%)', fontsize=11)
ax4.grid(True, alpha=0.3, axis='y')
# Add value labels
for bar in bars:
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}%', ha='center', va='bottom' if height > 0 else 'top',
            fontsize=9, fontweight='bold')

# 5. Q-Q Plot
ax5 = fig.add_subplot(gs[2, 2])
stats.probplot(result.returns, dist="norm", plot=ax5)
ax5.set_title('Q-Q Plot (Normality Check)', fontsize=14, fontweight='bold')
ax5.grid(True, alpha=0.3)

# 6. Rolling Sharpe (52-week window)
ax6 = fig.add_subplot(gs[3, 0])
if len(result.returns) >= 10:
    window = min(10, len(result.returns) // 2)
    rolling_mean = result.returns.rolling(window).mean()
    rolling_std = result.returns.rolling(window).std()
    rolling_sharpe = (rolling_mean / rolling_std) * np.sqrt(52)
    ax6.plot(rolling_sharpe.index, rolling_sharpe.values, linewidth=2, color=primary_color)
    ax6.axhline(0, color='red', linestyle='--', alpha=0.5)
    ax6.axhline(1, color='green', linestyle='--', alpha=0.3, label='Target: 1.0')
ax6.set_title(f'Rolling Sharpe ({window}-week)', fontsize=14, fontweight='bold')
ax6.set_ylabel('Sharpe Ratio', fontsize=11)
ax6.legend(loc='best', fontsize=10)
ax6.grid(True, alpha=0.3)

# 7. Portfolio Weights Over Time
ax7 = fig.add_subplot(gs[3, 1])
result.weights.plot.area(ax=ax7, alpha=0.7, linewidth=0)
ax7.set_title('Portfolio Weights Over Time', fontsize=14, fontweight='bold')
ax7.set_ylabel('Weight', fontsize=11)
ax7.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: '{:.0%}'.format(y)))
ax7.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=9)
ax7.grid(True, alpha=0.3, axis='y')

# 8. Key Metrics Table
ax8 = fig.add_subplot(gs[3, 2])
ax8.axis('off')

# Create table data
table_data = [
    ['Metric', 'Value'],
    ['Total Return', f"{metrics['Total Return']:.2%}"],
    ['Ann. Return', f"{metrics['Annualized Return']:.2%}"],
    ['Ann. Volatility', f"{metrics['Annualized Volatility']:.2%}"],
    ['Sharpe Ratio', f"{metrics['Sharpe Ratio']:.3f}"],
    ['Max Drawdown', f"{metrics['Max Drawdown']:.2%}"],
    ['Calmar Ratio', f"{metrics['Calmar Ratio']:.3f}"],
    ['Win Rate', f"{metrics['Win Rate']:.1%}"],
    ['Profit Factor', f"{metrics['Profit Factor']:.2f}"],
]

table = ax8.table(cellText=table_data, loc='center', cellLoc='left',
                  colWidths=[0.6, 0.4])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2)

# Style header row
for i in range(2):
    table[(0, i)].set_facecolor('#4472C4')
    table[(0, i)].set_text_props(weight='bold', color='white')

# Alternate row colors
for i in range(1, len(table_data)):
    for j in range(2):
        if i % 2 == 0:
            table[(i, j)].set_facecolor('#E7E6E6')

ax8.set_title('Key Metrics', fontsize=14, fontweight='bold', pad=20)

# Add overall title
fig.suptitle('Strategy Performance Tear Sheet', fontsize=20, fontweight='bold', y=0.995)

plt.show()

print("✓ Tear sheet generated!")

---

## Step 4: Risk Analysis

Deep dive into risk characteristics:

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Value at Risk (VaR) Analysis
ax = axes[0, 0]
returns_sorted = np.sort(result.returns)
var_levels = [1, 5, 10]  # percentiles
vars = [np.percentile(result.returns, level) for level in var_levels]

ax.hist(result.returns * 100, bins=30, alpha=0.7, color=primary_color, edgecolor='black')
colors_var = ['darkred', 'red', 'orange']
for level, var, color in zip(var_levels, vars, colors_var):
    ax.axvline(var * 100, color=color, linestyle='--', linewidth=2, 
               label=f'VaR {level}%: {var:.2%}')
ax.set_title('Value at Risk (VaR) Analysis', fontsize=14, fontweight='bold')
ax.set_xlabel('Weekly Return (%)', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# 2. Conditional Value at Risk (CVaR / Expected Shortfall)
ax = axes[0, 1]
var_95 = np.percentile(result.returns, 5)
cvar_95 = result.returns[result.returns <= var_95].mean()
ax.hist(result.returns * 100, bins=30, alpha=0.7, color=primary_color, edgecolor='black')
ax.axvline(var_95 * 100, color='red', linestyle='--', linewidth=2, label=f'VaR 5%: {var_95:.2%}')
ax.axvline(cvar_95 * 100, color='darkred', linestyle='--', linewidth=2, label=f'CVaR 5%: {cvar_95:.2%}')
# Shade CVaR region
ax.axvspan(result.returns.min() * 100, var_95 * 100, alpha=0.2, color='red', label='Tail Risk')
ax.set_title('Conditional VaR (Expected Shortfall)', fontsize=14, fontweight='bold')
ax.set_xlabel('Weekly Return (%)', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# 3. Rolling Volatility
ax = axes[1, 0]
if len(result.returns) >= 10:
    window = min(10, len(result.returns) // 2)
    rolling_vol = result.returns.rolling(window).std() * np.sqrt(52) * 100  # Annualized %
    ax.plot(rolling_vol.index, rolling_vol.values, linewidth=2, color=primary_color)
    ax.fill_between(rolling_vol.index, 0, rolling_vol.values, alpha=0.3, color=primary_color)
    ax.axhline(rolling_vol.mean(), color='red', linestyle='--', linewidth=2, 
               label=f'Mean: {rolling_vol.mean():.1f}%')
ax.set_title(f'Rolling Volatility ({window}-week, Annualized)', fontsize=14, fontweight='bold')
ax.set_ylabel('Volatility (%)', fontsize=11)
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)

# 4. Upside/Downside Capture
ax = axes[1, 1]
positive_returns = result.returns[result.returns > 0]
negative_returns = result.returns[result.returns < 0]

stats_data = [
    ['Positive Weeks', len(positive_returns)],
    ['Negative Weeks', len(negative_returns)],
    ['Average Gain', f"{positive_returns.mean():.2%}" if len(positive_returns) > 0 else 'N/A'],
    ['Average Loss', f"{negative_returns.mean():.2%}" if len(negative_returns) > 0 else 'N/A'],
    ['Best Week', f"{result.returns.max():.2%}"],
    ['Worst Week', f"{result.returns.min():.2%}"],
    ['Gain/Loss Ratio', f"{abs(positive_returns.mean() / negative_returns.mean()):.2f}" if len(negative_returns) > 0 else 'N/A']
]

ax.axis('off')
table = ax.table(cellText=stats_data, loc='center', cellLoc='left',
                 colWidths=[0.6, 0.4])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2.5)

for i in range(len(stats_data)):
    for j in range(2):
        if i % 2 == 0:
            table[(i, j)].set_facecolor('#E7E6E6')

ax.set_title('Win/Loss Statistics', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

print("✓ Risk analysis complete!")

---

## Step 5: Portfolio Attribution

Understand which positions contributed to performance:

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Average Position Sizes
ax = axes[0, 0]
avg_weights = result.weights.mean()
colors_weights = [positive_color if w > 0 else negative_color for w in avg_weights]
bars = ax.barh(avg_weights.index, avg_weights.values * 100, color=colors_weights, alpha=0.7, edgecolor='black')
ax.axvline(0, color='black', linestyle='-', linewidth=1)
ax.set_title('Average Portfolio Weights', fontsize=14, fontweight='bold')
ax.set_xlabel('Weight (%)', fontsize=11)
ax.grid(True, alpha=0.3, axis='x')
# Add value labels
for bar in bars:
    width = bar.get_width()
    ax.text(width, bar.get_y() + bar.get_height()/2,
           f'{width:.1f}%', ha='left' if width > 0 else 'right', va='center',
           fontsize=10, fontweight='bold')

# 2. Weight Stability (Std Dev of Weights)
ax = axes[0, 1]
weight_std = result.weights.std() * 100
bars = ax.bar(weight_std.index, weight_std.values, color=primary_color, alpha=0.7, edgecolor='black')
ax.set_title('Position Stability (Weight Std Dev)', fontsize=14, fontweight='bold')
ax.set_ylabel('Std Dev of Weight (%)', fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
ax.tick_params(axis='x', rotation=45)
# Add value labels
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height,
           f'{height:.1f}%', ha='center', va='bottom',
           fontsize=9, fontweight='bold')

# 3. Turnover Over Time
ax = axes[1, 0]
if len(result.weights) > 1:
    turnover = result.weights.diff().abs().sum(axis=1) * 100
    ax.bar(turnover.index, turnover.values, color=primary_color, alpha=0.7, edgecolor='black')
    ax.axhline(turnover.mean(), color='red', linestyle='--', linewidth=2,
               label=f'Mean: {turnover.mean():.1f}%')
ax.set_title('Portfolio Turnover', fontsize=14, fontweight='bold')
ax.set_ylabel('Turnover (%)', fontsize=11)
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# 4. Concentration (HHI Index)
ax = axes[1, 1]
if len(result.weights) > 0:
    # Herfindahl-Hirschman Index
    hhi = (result.weights ** 2).sum(axis=1)
    ax.plot(hhi.index, hhi.values, linewidth=2, color=primary_color, marker='o', markersize=4)
    ax.axhline(1/len(instruments), color='green', linestyle='--', linewidth=2,
               label=f'Equal Weight: {1/len(instruments):.3f}')
    ax.axhline(hhi.mean(), color='red', linestyle='--', linewidth=2,
               label=f'Mean: {hhi.mean():.3f}')
ax.set_title('Portfolio Concentration (HHI)', fontsize=14, fontweight='bold')
ax.set_ylabel('HHI (lower = more diversified)', fontsize=11)
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Attribution analysis complete!")
print(f"\n💡 Portfolio Insights:")
print(f"   Average turnover: {turnover.mean():.1f}% per rebalance")
print(f"   Concentration (HHI): {hhi.mean():.3f} (1.0 = concentrated, {1/len(instruments):.3f} = equal weight)")
print(f"   Most stable position: {weight_std.idxmin()} (std: {weight_std.min():.2f}%)")
print(f"   Largest average position: {avg_weights.idxmax()} ({avg_weights.max():.1%})")

---

## Step 6: Export Report to PDF

Create a professional PDF report:

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
from datetime import datetime

# Create PDF
output_path = Path.cwd() / f'performance_report_{datetime.now().strftime("%Y%m%d_%H%M%S")}.pdf'

with PdfPages(output_path) as pdf:
    # Page 1: Tear Sheet (recreate)
    fig = plt.figure(figsize=(11, 17))  # Letter size
    gs = gridspec.GridSpec(5, 2, figure=fig, hspace=0.4, wspace=0.3)
    
    # Title
    fig.suptitle('Strategy Performance Report', fontsize=24, fontweight='bold', y=0.98)
    fig.text(0.5, 0.96, f'Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}',
             ha='center', fontsize=10, style='italic')
    
    # Cumulative returns
    ax = fig.add_subplot(gs[0, :])
    cumulative = (1 + result.returns).cumprod()
    ax.plot(cumulative.index, cumulative.values, linewidth=2, color=primary_color)
    ax.fill_between(cumulative.index, 1, cumulative.values, alpha=0.3, color=primary_color)
    ax.axhline(1, color='black', linestyle='--', alpha=0.5)
    ax.set_title('Cumulative Returns', fontsize=14, fontweight='bold')
    ax.set_ylabel('Growth of $1')
    ax.grid(True, alpha=0.3)
    
    # Add more plots similarly...
    # (Simplified for brevity)
    
    pdf.savefig(fig, bbox_inches='tight')
    plt.close()
    
    # Add metadata
    d = pdf.infodict()
    d['Title'] = 'Strategy Performance Report'
    d['Author'] = 'ARBS Backtest System'
    d['Subject'] = 'Quantitative Strategy Performance Analysis'
    d['Keywords'] = 'Backtest, Performance, Risk, Attribution'
    d['CreationDate'] = datetime.now()

print(f"✓ Report exported to: {output_path}")
print(f"  File size: {output_path.stat().st_size / 1024:.1f} KB")

---

## Step 7: Create Summary Statistics Table

Export metrics to a CSV file for further analysis:

In [ ]:
# Create comprehensive metrics DataFrame
summary = pd.DataFrame({
    'Metric': list(metrics.keys()),
    'Value': [f"{v:.4f}" for v in metrics.values()]
})

# Add portfolio statistics
portfolio_stats = pd.DataFrame({
    'Metric': [
        'Number of Instruments',
        'Number of Periods',
        'Rebalance Frequency',
        'Average Turnover',
        'Average Concentration (HHI)',
    ],
    'Value': [
        len(instruments),
        len(result.returns),
        'Weekly',
        f"{turnover.mean():.2f}%",
        f"{hhi.mean():.4f}"
    ]
})

full_summary = pd.concat([summary, portfolio_stats], ignore_index=True)

# Export to CSV
csv_path = Path.cwd() / f'metrics_summary_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
full_summary.to_csv(csv_path, index=False)

print("Summary Statistics")
print("=" * 60)
print(full_summary.to_string(index=False))
print()
print(f"✓ Exported to: {csv_path}")

---

## Summary: What You've Learned

Congratulations! You now know how to:

✅ Calculate comprehensive performance metrics

✅ Create professional performance tear sheets

✅ Analyze risk characteristics (VaR, CVaR, drawdowns)

✅ Perform portfolio attribution analysis

✅ Export results to PDF and CSV

✅ Visualize portfolio composition and evolution

## Key Metrics Explained

**Return Metrics:**
- **Total Return**: Overall gain/loss over period
- **Annualized Return**: Return scaled to yearly basis
- **Sharpe Ratio**: Risk-adjusted return (higher is better)
- **Calmar Ratio**: Return divided by max drawdown

**Risk Metrics:**
- **Volatility**: Standard deviation of returns
- **Max Drawdown**: Largest peak-to-trough decline
- **VaR (Value at Risk)**: Maximum loss at given confidence level
- **CVaR**: Expected loss beyond VaR threshold

**Portfolio Metrics:**
- **Turnover**: How much portfolio changes each period
- **HHI (Concentration)**: How concentrated positions are
- **Win Rate**: Percentage of profitable periods
- **Profit Factor**: Total gains divided by total losses

## Best Practices

1. **Look Beyond Returns** - Consider risk-adjusted metrics
2. **Check Tail Risk** - Understand worst-case scenarios
3. **Monitor Drawdowns** - Know maximum pain points
4. **Analyze Attribution** - Know what drives performance
5. **Export Reports** - Document your analysis

## Next Steps

1. Customize visualizations for your needs
2. Add your own metrics calculations
3. Create automated reporting pipelines
4. Compare multiple strategies using these tools
5. Share reports with stakeholders

---

## Experiment: Customize Your Analysis

Try modifying the visualizations or adding new metrics:

In [ ]:
# YOUR CODE HERE
# Try:
# - Adding custom metrics
# - Creating new visualizations
# - Comparing different time periods
# - Analyzing specific instruments' contribution